# SQL — What, Why, When?

**What?** SQL (Structured Query Language) is used to read and manage data stored in big databases.

**Why?** To filter, combine and summarize data without loading everything into Python.

**When?** When data is stored in tables, grows large, or needs repeatable queries.



**What to use?**

| Task | SQL |
|---|---|
| Choose columns | `SELECT` |
| Filter rows | `WHERE` |
| Sort results | `ORDER BY` |
| Summarize | `GROUP BY`, `AVG`, `COUNT` |
| Combine tables | `JOIN` |
| Reuse a query step | `WITH` (Common table expression) |

## Lesson 1

Database: `lesson1.db`. See [README](README.md) for setup.

In [1]:
import sqlite3
import pandas as pd

# Connect to database
conn = sqlite3.connect("lesson1.db")

# Run SQL and display results
def sql(query):
    display(pd.read_sql_query(query, conn))

# SQL Essentials

**Think of SQL as asking questions about a table.**

Example table: `experiments`

| id | treatment | day | mineral |
|---:|---|---:|---:|
| 1 | Control | 7 | 10 |
| 2 | Drug_A | 7 | 20 |
| 3 | Control | 14 | 15 |
| 4 | Drug_A | 14 | 30 |
| 5 | Control | 21 | 19 |
| 6 | Drug_A | 21 | 38 |
| 7 | Drug_B | 7 | 12 |
| 8 | Drug_B | 14 | 22 |
| 9 | Drug_B | 21 | 29 |
| 10 | Control | 28 | 24 |
| 11 | Drug_A | 28 | 45 |
| 12 | Drug_B | 28 | 33 |



# 1. Get data — `SELECT`

In [2]:
sql("""
SELECT treatment, mineral
FROM experiments;
""")

,treatment,mineral
0,Control,10
1,Drug_A,20
2,Control,15
3,Drug_A,30
4,Control,19
5,Drug_A,38
6,Drug_B,12
7,Drug_B,22
8,Drug_B,29
9,Control,24


Use `SELECT *` to get **all columns**.



# 2. Filter rows — `WHERE`

In [3]:
sql("""
SELECT *
FROM experiments
WHERE day = 14 AND mineral > 20;
""")

,id,treatment,day,mineral
0,4,Drug_A,14,30
1,8,Drug_B,14,22


# REMEMBER

### **SQL Comparison and Logical Operators**

| Operator   | Meaning                        | Example                 |
| ---------- | ------------------------------ | ----------------------- |
| `=`        | Equal to                       | `age = 25`              |
| `!=`       | Not equal to                   | `age != 25`             |
| `>`        | Greater than                   | `age > 25`              |
| `<`        | Less than                      | `age < 25`              |
| `>=`       | Greater than or equal          | `age >= 25`             |
| `<=`       | Less than or equal             | `age <= 25`             |
| `AND`      | Both conditions must be true   | `age > 20 AND age < 40` |
| `OR`       | At least one condition is true | `age < 20 OR age > 40`  |
| `IN (...)` | Matches any value in a list    | `age IN (20, 25, 30)`   |



# 3. Sort and limit — `ORDER BY`, `LIMIT`

In [4]:
sql("""
SELECT *
FROM experiments
ORDER BY mineral DESC
LIMIT 2;
""")

,id,treatment,day,mineral
0,11,Drug_A,28,45
1,6,Drug_A,21,38


## `DESC` = highest first; `ASC` = lowest first.

### Limit decides how many rows to show.

# 4. Calculate summaries — `GROUP BY`

In [5]:
sql("""
SELECT treatment,
       COUNT(*) AS n,
       AVG(mineral) AS mean_mineral
FROM experiments
GROUP BY treatment;
""")

,treatment,n,mean_mineral
0,Control,4,17.00
1,Drug_A,4,33.25
2,Drug_B,4,24.00


Logic:
1. FROM → Read the experiments table.
2. GROUP BY treatment → Separate rows by treatment (Control, Drug_A, etc.).
3. COUNT(*) → Count experiments in each group.
4. AVG(mineral) → Calculate mean mineralization per group.
5. AS → Give the output columns readable names.

Useful functions: `COUNT`, `AVG`, `SUM`, `MIN`, `MAX`.

Use `WHERE` **before grouping**; use `HAVING` to **filter groups**:

In [6]:
sql(""" 
SELECT treatment, AVG(mineral) AS mean_mineral
FROM experiments
GROUP BY treatment
HAVING AVG(mineral) > 15;
""")

,treatment,mean_mineral
0,Control,17.00
1,Drug_A,33.25
2,Drug_B,24.00


# 5. Combine tables — `JOIN`

Imagine another table: `samples(id, sample_name)`.

In [7]:
sql("""
SELECT e.treatment, s.sample_name
FROM experiments AS e
LEFT JOIN samples AS s
    ON e.id = s.id;
""")

,treatment,sample_name
0,Control,A1
1,Drug_A,A2
2,Control,A3
3,Drug_A,A4
4,Control,A5
5,Drug_A,A6
6,Drug_B,A7
7,Drug_B,A8
8,Drug_B,A9
9,Control,A10


Logic
1. FROM experiments AS e → Start with the experiments table. Rename it e for convenience.
2. LEFT JOIN samples AS s → Connect the samples table, renamed s.
3. ON e.id = s.id → Match rows where the IDs are equal.
4. SELECT e.treatment, s.sample_name → Return these two columns.


- `INNER JOIN`: only matching rows.
- `LEFT JOIN`: all left-table rows, even without matches.



# 6. Add categories — `CASE`

In [8]:
sql("""
SELECT mineral,
       CASE
           WHEN mineral >= 20 THEN 'High'
           ELSE 'Low'
       END AS category
FROM experiments;
""")

,mineral,category
0,10,Low
1,20,High
2,15,Low
3,30,High
4,19,Low
5,38,High
6,12,Low
7,22,High
8,29,High
9,24,High


Logic — CASE WHEN (like if/else in Python):
1. SELECT mineral → Retrieve mineral values.
2. CASE → Start a conditional check.
3. WHEN mineral >= 20 THEN 'High' → Label values ≥20 as High.
4. ELSE 'Low' → Label everything else as Low.
5. END AS category → Create a new column called category.

# 7. Break up a query — `WITH` (CTE)

In [9]:
sql("""
WITH averages AS (
    SELECT treatment, AVG(mineral) AS mean_mineral
    FROM experiments
    GROUP BY treatment
)
SELECT *
FROM averages
WHERE mean_mineral > 15;
""")

,treatment,mean_mineral
0,Control,17.00
1,Drug_A,33.25
2,Drug_B,24.00


Logic — WITH (Common Table Expression / CTE):
1. WITH averages AS (...) → Create a temporary named result called averages.
2. SELECT treatment, AVG(mineral) → Calculate mean mineralization for each treatment.
3. GROUP BY treatment → Group experiments by treatment.
4. SELECT * FROM averages → Retrieve the calculated results.
5. WHERE mean_mineral > 15 → Keep only treatments with an average above 15.

## The pattern to remember

```sql
SELECT column, AVG(value)
FROM table_name
WHERE condition
GROUP BY column
HAVING AVG(value) > 10
ORDER BY column
LIMIT 5;
```



**Learn in this order:** `SELECT` → `WHERE` → `ORDER BY` → `GROUP BY` → `JOIN` → `CASE` → `WITH`.
